# Quadrant Classification: CRISPR vs RNA-seq

This notebook plots a quadrant chart:
- X-axis: RNA-seq |log2FC| (absolute value)
- Y-axis: CRISPR Score (gene-level log2FC or other scores; higher values indicate stronger enrichment)

Tasks:
- Clearly label the position of `ANPEP` (representative of high Y, low X).
- Highlight genes in the top-right quadrant (strong CRISPR enrichment + significant expression change) and export the gene list.

## Assumed Input File Format (CSV/TSV)
Assumes a merged table with the following columns (can be remapped via parameters):
- `gene_id`: Optional, gene ID (e.g., ENSSSCG...).
- `gene_symbol`: Gene symbol (e.g., ANPEP).
- `crispr_score`: CRISPR gene-level score (e.g., median log2FC across multiple sgRNAs per gene).
- `rna_log2fc`: RNA-seq differential expression log2FC (Treatment vs Control).
- `rna_padj`: Optional, RNA-seq FDR/q-value (Benjamini-Hochberg).

Example file header (CSV):
```
gene_id,gene_symbol,crispr_score,rna_log2fc,rna_padj
ENSSSCG00000011811,ANPEP,15.58,-0.20,0.08
...
```



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✓ imports ready')


In [ ]:
# === Parameters ===
# Input file (CSV or TSV). Default example path; modify as needed.
INPUT_FILE = '/path/to/CRISPR/results/crispr_rnaseq_merged.csv'
DELIM = ','  # Change to '\t' for TSV

# Column name mapping (modify here if your column names differ)
COL_GENE_ID = 'gene_id'
COL_SYMBOL = 'gene_symbol'
COL_CRISPR = 'crispr_score'   # Y-axis
COL_RNA_L2FC = 'rna_log2fc'   # X-axis (absolute value)
COL_RNA_PADJ = 'rna_padj'     # Optional

# Thresholds (adjustable)
Y_THRESHOLD = 2.0    # CRISPR strong enrichment threshold (e.g., gene-level log2FC >= 2)
X_THRESHOLD = 1.0    # Significant expression change threshold (|RNA log2FC| >= 1)
PADJ_THRESHOLD = 0.05  # Optional: expression significance FDR

# Visualization
FIGSIZE = (8, 6)
HIGHLIGHT_GENES = ['ANPEP', 'WDR13']  # Genes to specially mark

# CRISPR category list
CRISPR_CATEGORIES = ['Depleted', 'Slightly Enriched', 'Moderately Enriched', 'Highly Enriched']

# Scatter plot default parameters
SCATTER_DEFAULTS = {
    'alpha': 0.6,
    's': 80,  # Doubled size (from 40 to 80)
    'marker': 'o',
    'edgecolor': 'black',
    'linewidth': 1.0  # Increased border width
}

In [ ]:
# Load & validate
p = Path(INPUT_FILE)
if not p.exists():
    print(f'Warning: Input file does not exist: {p}')
else:
    df = pd.read_csv(p, sep=DELIM)
    print('Loaded:', p)
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))

    required = [COL_SYMBOL, COL_CRISPR, COL_RNA_L2FC]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    # Ensure numeric types
    df[COL_CRISPR] = pd.to_numeric(df[COL_CRISPR], errors='coerce')
    df[COL_RNA_L2FC] = pd.to_numeric(df[COL_RNA_L2FC], errors='coerce')
    if COL_RNA_PADJ in df.columns:
        df[COL_RNA_PADJ] = pd.to_numeric(df[COL_RNA_PADJ], errors='coerce')

    # Derived quantity: |RNA log2FC|
    df['rna_log2fc_abs'] = df[COL_RNA_L2FC].abs()

    # Select data for plotting
    plot_df = df.dropna(subset=[COL_CRISPR, 'rna_log2fc_abs']).copy()
    print('Usable rows:', len(plot_df))

In [ ]:
# Define plotting function
def plot_quadrant_scatter(plot_df, ax, save_mode=False):
    """
    Draw quadrant scatter plot.
    
    Parameters:
    -----------
    plot_df : DataFrame
        DataFrame containing plot data
    ax : matplotlib.axes.Axes
        Axes object for plotting
    save_mode : bool
        Whether in save mode (save mode does not show label adjustments)
    """
    # Define CRISPR category colors (blue-red-orange-yellow color scheme)
    CRISPR_COLORS = {
        'Depleted': '#4472C4',              # Blue (Depleted)
        'Slightly Enriched': '#FFD700',     # Yellow (Slightly Enriched)
        'Moderately Enriched': '#FF8C00',  # Orange (Moderately Enriched)
        'Highly Enriched': '#DC143C',      # Red (Highly Enriched)
    }
    
    # Define highlighted gene annotation colors (for text annotations)
    GENE_ANNOTATION_COLORS = {
        'ANPEP': 'orange',
        'WDR13': 'blue'
    }
    
    # Plot scatter points by CRISPR category
    if 'crispr_category' in plot_df.columns:
        for category in CRISPR_CATEGORIES:
            cat_df = plot_df[plot_df['crispr_category'] == category]
            if not cat_df.empty:
                ax.scatter(
                    cat_df['rna_log2fc_abs'], cat_df[COL_CRISPR],
                    color=CRISPR_COLORS[category],
                    **SCATTER_DEFAULTS,
                    label=category
                )
    else:
        ax.scatter(
            plot_df['rna_log2fc_abs'], plot_df[COL_CRISPR],
            color='#bdbdbd', **SCATTER_DEFAULTS
        )
    
    # Top-right points (using thicker border)
    top_df = plot_df[plot_df['top_right']]
    if not top_df.empty:
        for category in CRISPR_CATEGORIES:
            if 'crispr_category' in top_df.columns:
                cat_top_df = top_df[top_df['crispr_category'] == category]
                if not cat_top_df.empty:
                    ax.scatter(
                        cat_top_df['rna_log2fc_abs'], cat_top_df[COL_CRISPR],
                        color=CRISPR_COLORS[category],
                        alpha=SCATTER_DEFAULTS['alpha'],
                        s=SCATTER_DEFAULTS['s'],
                        marker=SCATTER_DEFAULTS['marker'],
                        edgecolor='black',  # Ensure black border
                        linewidth=2.0,  # Thicker border
                        zorder=10
                    )
            else:
                ax.scatter(
                    top_df['rna_log2fc_abs'], top_df[COL_CRISPR],
                    color='#bdbdbd',
                    alpha=SCATTER_DEFAULTS['alpha'],
                    s=SCATTER_DEFAULTS['s'],
                    marker=SCATTER_DEFAULTS['marker'],
                    edgecolor='black',  # Ensure black border
                    linewidth=2.0,  # Thicker border
                    zorder=10
                )
                break
        
        # Label top-right gene names (except specially highlighted genes)
        if not save_mode:
            from adjustText import adjust_text
            texts = []
            for _, r in top_df.iterrows():
                if r[COL_SYMBOL].upper() in [g.upper() for g in HIGHLIGHT_GENES]:
                    continue
                texts.append(
                    ax.text(
                        r['rna_log2fc_abs'], r[COL_CRISPR], r[COL_SYMBOL],
                        fontsize=9, fontweight='bold', color='darkred',
                        ha='center', va='bottom', zorder=15
                    )
                )
            try:
                adjust_text(
                    texts,
                    arrowprops=dict(arrowstyle='-', color='red', lw=0.5, alpha=0.7),
                    ax=ax
                )
            except:
                pass
    
    # Specially annotate highlighted genes (color consistent with their category)
    for gene in HIGHLIGHT_GENES:
        col_name = f'is_{gene.lower()}'
        if col_name in plot_df.columns:
            gene_df = plot_df[plot_df[col_name]]
            if not gene_df.empty:
                for _, r in gene_df.iterrows():
                    # Determine color based on category
                    if 'crispr_category' in plot_df.columns and r['crispr_category'] in CRISPR_COLORS:
                        gene_color = CRISPR_COLORS[r['crispr_category']]
                    else:
                        gene_color = '#bdbdbd'  # Default gray
                    
                    # Get annotation color
                    annotation_color = GENE_ANNOTATION_COLORS.get(gene.upper(), 'black')
                    
                    ax.scatter(
                        r['rna_log2fc_abs'], r[COL_CRISPR],
                        marker=SCATTER_DEFAULTS['marker'],
                        s=SCATTER_DEFAULTS['s'],  # Uniform size
                        color=gene_color,  # Use category color
                        edgecolor=SCATTER_DEFAULTS['edgecolor'],
                        linewidth=2.0,  # Slightly thicker border for emphasis
                        label=gene,
                        zorder=20
                    )
                    if not save_mode:
                        ax.annotate(
                            gene,
                            xy=(r['rna_log2fc_abs'], r[COL_CRISPR]),
                            xytext=(15, 15),
                            textcoords='offset points',
                            fontsize=11,
                            fontweight='bold',
                            color=annotation_color,
                            bbox=dict(boxstyle='round,pad=0.5', fc=gene_color, ec=annotation_color, lw=2, alpha=0.95),
                            arrowprops=dict(arrowstyle='->', color=annotation_color, lw=2)
                        )
    
    # Threshold lines and annotations
    ax.axvline(X_THRESHOLD, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, zorder=1)
    ax.axhline(Y_THRESHOLD, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, zorder=1)
    ax.text(X_THRESHOLD * 1.1, ax.get_ylim()[1] * 0.95,
            f'|log2FC| >= {X_THRESHOLD}',
            fontsize=9, color='gray', style='italic')
    ax.text(ax.get_xlim()[1] * 0.02, Y_THRESHOLD * 1.1,
            f'CRISPR >= {Y_THRESHOLD}',
            fontsize=9, color='gray', style='italic')
    
    # Set labels and legend
    ax.set_xlabel('|RNA-seq log2FC|', fontsize=12, fontweight='bold')
    ax.set_ylabel('CRISPR Score (gene-level log2FC)', fontsize=12, fontweight='bold')
    ax.set_title('Quadrant Classification: CRISPR vs RNA-seq', fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='upper right', frameon=True, framealpha=0.9, fontsize=9,
              title='CRISPR Category', title_fontsize=10)

print('✓ Plotting function defined')

In [ ]:
# Derive quadrant classification
if 'plot_df' not in globals():
    print('Warning: No plot_df data; the previous cell may not have loaded successfully.')
else:
    # Whether RNA significance is needed
    if COL_RNA_PADJ in plot_df.columns:
        plot_df['rna_sig'] = plot_df[COL_RNA_PADJ] < PADJ_THRESHOLD
    else:
        plot_df['rna_sig'] = True  # No padj provided, no restriction

    # Top-right (strong enrichment + significant expression change)
    plot_df['top_right'] = (
        (plot_df[COL_CRISPR] >= Y_THRESHOLD) &
        (plot_df['rna_log2fc_abs'] >= X_THRESHOLD) &
        (plot_df['rna_sig'])
    )

    # Mark genes to highlight
    for gene in HIGHLIGHT_GENES:
        col_name = f'is_{gene.lower()}'
        plot_df[col_name] = plot_df[COL_SYMBOL].astype(str).str.upper() == gene.upper()

    print('Top-right genes:', int(plot_df['top_right'].sum()))
    
    # Print highlighted gene information
    print('\nHighlighted gene information:')
    for gene in HIGHLIGHT_GENES:
        gene_df = plot_df[plot_df[COL_SYMBOL].astype(str).str.upper() == gene.upper()]
        if not gene_df.empty:
            r = gene_df.iloc[0]
            print(f"  {gene}: CRISPR={r[COL_CRISPR]:.2f}, RNA-log2FC={r[COL_RNA_L2FC]:.2f}, |log2FC|={r['rna_log2fc_abs']:.2f}")
        else:
            print(f"  {gene}: Not found")
    
    plot_df.head()

In [ ]:
# Plot with CRISPR category colors
if 'plot_df' not in globals():
    print('Warning: No plot_df data; cannot plot.')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    plot_quadrant_scatter(plot_df, ax, save_mode=False)
    plt.tight_layout()
    plt.show()
    
    # Print top-right gene list
    top_df = plot_df[plot_df['top_right']]
    print(f'\nTop-right quadrant genes ({len(top_df)} total):')
    for _, r in top_df.iterrows():
        print(f"  - {r[COL_SYMBOL]}: CRISPR={r[COL_CRISPR]:.2f}, |RNA-log2FC|={r['rna_log2fc_abs']:.2f}")

In [ ]:
# Save outputs
RESULTS_DIR = '/path/to/CRISPR/results'
PREFIX = 'Quadrant_Classification'

if 'plot_df' in globals():
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    
    # Save figures
    fig, ax = plt.subplots(figsize=(10, 7))
    plot_quadrant_scatter(plot_df, ax, save_mode=True)
    plt.tight_layout()
    
    fig.savefig(f"{RESULTS_DIR}/{PREFIX}_scatter.png", dpi=300, bbox_inches='tight')
    fig.savefig(f"{RESULTS_DIR}/{PREFIX}_scatter.pdf", bbox_inches='tight')
    plt.close(fig)
    print(f'✓ Saved figures: {PREFIX}_scatter.png/pdf')
    
    # Top-right gene list (including crispr_category)
    cols_to_export = [COL_GENE_ID if COL_GENE_ID in plot_df.columns else COL_SYMBOL, 
                      COL_SYMBOL, COL_CRISPR, 'rna_log2fc_abs']
    if 'crispr_category' in plot_df.columns:
        cols_to_export.append('crispr_category')
    if 'rna_deg_status' in plot_df.columns:
        cols_to_export.append('rna_deg_status')
    if COL_RNA_L2FC in plot_df.columns:
        cols_to_export.append(COL_RNA_L2FC)
        
    top_genes = plot_df.loc[plot_df['top_right'], cols_to_export].copy()
    top_genes = top_genes.sort_values(COL_CRISPR, ascending=False)
    out_csv = f"{RESULTS_DIR}/{PREFIX}_top_right_genes.csv"
    top_genes.to_csv(out_csv, index=False)
    print(f'✓ Exported: {out_csv}')
    print(f'\nExported top-right genes:')
    for _, r in top_genes.iterrows():
        print(f"  {r[COL_SYMBOL]}: CRISPR={r[COL_CRISPR]:.2f}, |log2FC|={r['rna_log2fc_abs']:.2f}")
else:
    print('Warning: No plot_df; skipping export.')